In [25]:
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

from dotenv import load_dotenv
load_dotenv()
embedding_function = OpenAIEmbeddings(model="text-embedding-3-large")

from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [26]:
docs = [
    Document(
        page_content="Peak Performance Gym was founded in 2015 by former Olympic athlete Marcus Chen. With over 15 years of experience in professional athletics, Marcus established the gym to provide personalized fitness solutions for people of all levels. The gym spans 10,000 square feet and features state-of-the-art equipment.",
        metadata={"source": "about.txt"}
    ),
    Document(
        page_content="Peak Performance Gym is open Monday through Friday from 5:00 AM to 11:00 PM. On weekends, our hours are 7:00 AM to 9:00 PM. We remain closed on major national holidays. Members with Premium access can enter using their key cards 24/7, including holidays.",
        metadata={"source": "hours.txt"}
    ),
    Document(
        page_content="Our membership plans include: Basic (₹1,500/month) with access to gym floor and basic equipment; Standard (₹2,500/month) adds group classes and locker facilities; Premium (₹4,000/month) includes 24/7 access, personal training sessions, and spa facilities. We offer student and senior citizen discounts of 15% on all plans. Corporate partnerships are available for companies with 10+ employees joining.",
        metadata={"source": "membership.txt"}
    ),
    Document(
        page_content="Group fitness classes at Peak Performance Gym include Yoga (beginner, intermediate, advanced), HIIT, Zumba, Spin Cycling, CrossFit, and Pilates. Beginner classes are held every Monday and Wednesday at 6:00 PM. Intermediate and advanced classes are scheduled throughout the week. The full schedule is available on our mobile app or at the reception desk.",
        metadata={"source": "classes.txt"}
    ),
    Document(
        page_content="Personal trainers at Peak Performance Gym are all certified professionals with minimum 5 years of experience. Each new member receives a complimentary fitness assessment and one free session with a trainer. Our head trainer, Neha Kapoor, specializes in rehabilitation fitness and sports-specific training. Personal training sessions can be booked individually (₹800/session) or in packages of 10 (₹7,000) or 20 (₹13,000).",
        metadata={"source": "trainers.txt"}
    ),
    Document(
        page_content="Peak Performance Gym's facilities include a cardio zone with 30+ machines, strength training area, functional fitness space, dedicated yoga studio, spin class room, swimming pool (25m), sauna and steam rooms, juice bar, and locker rooms with shower facilities. Our equipment is replaced or upgraded every 3 years to ensure members have access to the latest fitness technology.",
        metadata={"source": "facilities.txt"}
    )
]

In [27]:
new_db = FAISS.load_local("faiss_index", embedding_function,allow_dangerous_deserialization=True)
retriever = new_db.as_retriever(search_type="mmr", search_kwargs = {"k": 3})


In [28]:
from langchain.tools.retriever import create_retriever_tool
from langchain_core.tools import tool

retrievel_tool=create_retriever_tool(retriever,
                                    "retriever_tool",
    "Information related to Gym History & Founder, Operating Hours, Membership Plans, Fitness Classes, Personal Trainers, and Facilities & Equipment of Peak Performance Gym",
)

@tool
def off_topic():
    """Catch all Questions NOT related to Peak Performance Gym's history, hours, membership plans, fitness classes, trainers, or facilities"""
    return "Forbidden - do not respond to the user"

tools=[off_topic,retrievel_tool]

In [29]:
from typing import Annotated,Sequence,TypedDict,Literal
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.graph.message import add_messages

class Agentstate(TypedDict):
    messages:Annotated[Sequence[BaseMessage],add_messages]

In [60]:
from langgraph.graph import StateGraph,START,END

def Agent(state:Agentstate):
    print("-----start-----")

    message=state["messages"][-1].content
    model=ChatOpenAI()
    model=model.bind_tools(tools)
    response=model.invoke(message)
    return{
        "messages":[response]
    }

def should_continue(state)->Literal["tools",END]:
    message=state["messages"]
    last_msg=message[-1]
    print("lll",last_msg)
    if getattr(last_msg,"tool_calls",None):
        return "tools"
    return END



In [61]:
from langgraph.prebuilt import ToolNode

graph=StateGraph(Agentstate)
tool_node=ToolNode(tools)

graph.add_node("agent",Agent)
graph.set_entry_point("agent")
graph.add_node("tools",tool_node)
graph.add_conditional_edges("agent",
                            should_continue)

graph.add_edge("tools","agent")
app = graph.compile()

In [62]:
from langchain_core.messages import HumanMessage

app.invoke(
    input={"messages": [HumanMessage(content="How will the weather be tommorrow?")]}
)

-----start-----
lll content='' additional_kwargs={'tool_calls': [{'id': 'call_cwTFOW8rZGHmrSglN4HvYiwz', 'function': {'arguments': '{}', 'name': 'off_topic'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 122, 'total_tokens': 132, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CFfy1VMAynnW6EY77oLCsmwkmvjqA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--162dc982-ecdb-484c-a591-7c3881d3c552-0' tool_calls=[{'name': 'off_topic', 'args': {}, 'id': 'call_cwTFOW8rZGHmrSglN4HvYiwz', 'type': 'tool_call'}] usage_metadata={'input_tokens': 122, 'output_tokens': 10, 'total_tokens': 132, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_de

{'messages': [HumanMessage(content='How will the weather be tommorrow?', additional_kwargs={}, response_metadata={}, id='9e96c72d-464f-43a8-ae1d-72c9d7213f5d'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_cwTFOW8rZGHmrSglN4HvYiwz', 'function': {'arguments': '{}', 'name': 'off_topic'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 122, 'total_tokens': 132, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CFfy1VMAynnW6EY77oLCsmwkmvjqA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--162dc982-ecdb-484c-a591-7c3881d3c552-0', tool_calls=[{'name': 'off_topic', 'args': {}, 'id': 'call_cwTFOW8rZGHmrSglN4HvYiwz', 'type': 'tool_

In [59]:
app.invoke(input={"messages":[HumanMessage(content="Who is the owner and what are the timings?")]})

-----start-----
lll content='' additional_kwargs={'tool_calls': [{'id': 'call_hYFeXbZo1mcc0rLqU5LraMdC', 'function': {'arguments': '{"query": "owner of Peak Performance Gym"}', 'name': 'retriever_tool'}, 'type': 'function'}, {'id': 'call_UvfbPCtJAaZGx7SNEA9kyrMX', 'function': {'arguments': '{"query": "operating hours of Peak Performance Gym"}', 'name': 'retriever_tool'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 123, 'total_tokens': 180, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CFfuPzxzgUS2WiTLoTCPZc8rosso2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--da6d8649-dbbc-4b8e-8ad7-0f0a92778069-0' tool_calls=[{'name': 'retriever_tool', 'a

{'messages': [HumanMessage(content='Who is the owner and what are the timings?', additional_kwargs={}, response_metadata={}, id='ca098ef9-a64b-48a5-9585-6c5724bcf98e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_hYFeXbZo1mcc0rLqU5LraMdC', 'function': {'arguments': '{"query": "owner of Peak Performance Gym"}', 'name': 'retriever_tool'}, 'type': 'function'}, {'id': 'call_UvfbPCtJAaZGx7SNEA9kyrMX', 'function': {'arguments': '{"query": "operating hours of Peak Performance Gym"}', 'name': 'retriever_tool'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 123, 'total_tokens': 180, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CFfuPzxzgUS2WiTLoTCPZc8rosso2', 'se

In [2]:
l=[1,2]
s=None
s=len(l)>2
print(s)

False
